In [1]:
# %% [markdown]
# # 2. Transformación (Capa Silver)
# Lectura desde Bronze y preparación de datos

# %%
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.functions import col

# Crear SparkSession

master_url = "spark://spark-master:7077"

builder = (
    SparkSession.builder
    .appName("Lab_SECOP_Silver")
    .master(master_url)
    .config(
        "spark.jars.packages",
        "io.delta:delta-spark_2.12:3.0.0"
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config("spark.executor.memory", "2g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession capa plata iniciada")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3e95e9df-fbd5-47a6-a921-a99b6b8dd372;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 151ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   

SparkSession capa plata iniciada


In [2]:
# %%
# Lectura de la capa Bronze
# --------------------------------------------

bronze_path = "/app/data/lakehouse/bronze/secop"

print("Leyendo datos desde lakehouse/Bronze...")

df_bronze = spark.read.format("delta").load(bronze_path)

print(f"Registros en Bronze: {df_bronze.count()}")
df_bronze.printSchema()


Leyendo datos desde lakehouse/Bronze...


26/02/01 21:30:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/02/01 21:30:34 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
[Stage 3:======================================================>  (48 + 2) / 50]

Registros en Bronze: 100000
root
 |-- nombre_entidad: string (nullable = true)
 |-- nit_entidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- localizaci_n: string (nullable = true)
 |-- orden: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- rama: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- proceso_de_compra: string (nullable = true)
 |-- id_contrato: string (nullable = true)
 |-- referencia_del_contrato: string (nullable = true)
 |-- estado_contrato: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- tipo_de_contrato: string (nullable = true)
 |-- modalidad_de_contratacion: string (nullable = true)
 |-- justificacion_modalidad_de: string (nullable = true)
 |-- condiciones_de_entrega: string (nullable = true)
 |-- tipodocproveedor: string (nullable = true)
 |-- documento

In [3]:
# %%
# Selección de columnas relevantes para análisis

columnas_plata = [
    # Identificación del contrato
    "id_contrato",
    "referencia_del_contrato",
    "estado_contrato",

    # Entidad contratante
    "nombre_entidad",
    "nit_entidad",
    "departamento",
    "ciudad",
    "sector",
    "rama",
    "entidad_centralizada",

    # Proceso de contratación
    "tipo_de_contrato",
    "modalidad_de_contratacion",
    "codigo_de_categoria_principal",
    "descripcion_del_proceso",
    "objeto_del_contrato",

    # Fechas clave (base para análisis temporal)
    "fecha_de_firma",
    "fecha_de_inicio_del_contrato",
    "fecha_de_fin_del_contrato",

    # Valores económicos (core analítico)
    "valor_del_contrato",
    "valor_pagado",
    "valor_facturado",
    "valor_pendiente_de_pago",

    # Proveedor
    "proveedor_adjudicado",
    "documento_proveedor",
    "tipodocproveedor",

    # Metadata técnica (desde Bronze)
    "fecha_ingesta",
    "archivo_origen"
]

df_plata_base = df_bronze.select(
    *[col(c) for c in columnas_plata if c in df_bronze.columns]
)

print("Columnas seleccionadas para la capa Plata:")
df_plata_base.printSchema()
df_plata_base.show(5, truncate=False)

Columnas seleccionadas para la capa Plata:
root
 |-- id_contrato: string (nullable = true)
 |-- referencia_del_contrato: string (nullable = true)
 |-- estado_contrato: string (nullable = true)
 |-- nombre_entidad: string (nullable = true)
 |-- nit_entidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- rama: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- tipo_de_contrato: string (nullable = true)
 |-- modalidad_de_contratacion: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- objeto_del_contrato: string (nullable = true)
 |-- fecha_de_inicio_del_contrato: string (nullable = true)
 |-- fecha_de_fin_del_contrato: string (nullable = true)
 |-- valor_del_contrato: string (nullable = true)
 |-- valor_pagado: string (nullable = true)
 |-- valor_facturado: strin

+------------------+-----------------------+---------------+---------------------------------------------------------------------+-----------+--------------------------+------------+----------------------+---------+--------------------+-----------------------+-----------------------------+-----------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [4]:
# %%
from pyspark.sql import functions as F

# Cast de tipos (fechas y valores numéricos)
df_cast = (
    df_plata_base
    # Fechas
    .withColumn(
        "fecha_de_inicio_del_contrato",
        F.to_date("fecha_de_inicio_del_contrato")
    )
    .withColumn(
        "fecha_de_fin_del_contrato",
        F.to_date("fecha_de_fin_del_contrato")
    )

    # Valores monetarios
    .withColumn(
        "valor_del_contrato",
        F.col("valor_del_contrato").cast("double")
    )
    .withColumn(
        "valor_pagado",
        F.col("valor_pagado").cast("double")
    )
    .withColumn(
        "valor_facturado",
        F.col("valor_facturado").cast("double")
    )
    .withColumn(
        "valor_pendiente_de_pago",
        F.col("valor_pendiente_de_pago").cast("double")
    )
)

df_cast.printSchema()


root
 |-- id_contrato: string (nullable = true)
 |-- referencia_del_contrato: string (nullable = true)
 |-- estado_contrato: string (nullable = true)
 |-- nombre_entidad: string (nullable = true)
 |-- nit_entidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- rama: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- tipo_de_contrato: string (nullable = true)
 |-- modalidad_de_contratacion: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- objeto_del_contrato: string (nullable = true)
 |-- fecha_de_inicio_del_contrato: date (nullable = true)
 |-- fecha_de_fin_del_contrato: date (nullable = true)
 |-- valor_del_contrato: double (nullable = true)
 |-- valor_pagado: double (nullable = true)
 |-- valor_facturado: double (nullable = true)
 |-- valor_pendiente_de_pag

In [5]:
# %%
# Quality Gate + Clasificación (Silver / Quarantine)

df_validated = df_cast.withColumn(
    "_quality_checks",
    F.struct(
        # Regla 1: valor del contrato válido
        (F.col("valor_del_contrato") > 0).alias("valor_valido"),

        # Regla 2: fecha de inicio válida
        F.col("fecha_de_inicio_del_contrato").isNotNull().alias("fecha_valida")
    )
)

# Evaluación global
df_validated = df_validated.withColumn(
    "_is_valid",
    F.col("_quality_checks.valor_valido") &
    F.col("_quality_checks.fecha_valida")
)

# Motivo de rechazo
df_validated = df_validated.withColumn(
    "motivo_rechazo",
    F.when(
        ~F.col("_is_valid"),
        F.concat_ws(", ",
            F.when(~F.col("_quality_checks.valor_valido"),
                   "Valor del contrato <= 0"),
            F.when(~F.col("_quality_checks.fecha_valida"),
                   "Fecha inicio contrato nula")
        )
    )
)

# --------------------------------------------
# SPLIT: Silver limpio / Quarantine
# --------------------------------------------

df_silver_clean = df_validated.filter(F.col("_is_valid"))

df_quarantine = (
    df_validated
    .filter(~F.col("_is_valid"))
    .withColumn("fecha_cuarentena", F.current_timestamp())
)

print(f"✅ Registros Silver: {df_silver_clean.count()}")
print(f"❌ Registros Quarantine: {df_quarantine.count()}")


✅ Registros Silver: 7942
❌ Registros Quarantine: 92058


In [6]:
# %%
# --------------------------------------------
# Escritura capa Silver (registros válidos)
# --------------------------------------------
# --------------------------------------------
# Configuración para manejo de fechas antiguas
# (Requerido Spark 3.x + Parquet/Delta)
# --------------------------------------------

spark.conf.set(
    "spark.sql.parquet.datetimeRebaseModeInWrite",
    "LEGACY"
)

silver_path = "/app/data/lakehouse/silver/secop"

(
    df_silver_clean
    .drop("_quality_checks", "_is_valid")  # columnas técnicas no necesarias
    .write
    .format("delta")
    .mode("append")
    .save(silver_path)
)

print("✅ Datos escritos en capa Silver")
print(f"Ruta: {silver_path}")


[Stage 31:==========================================>             (38 + 2) / 50]

✅ Datos escritos en capa Silver
Ruta: /app/data/lakehouse/silver/secop


In [7]:
# Escritura capa Quarantine (registros inválidos)
quarantine_path = "/app/data/lakehouse/quarantine/secop_errors"

(
    df_quarantine
    .write
    .format("delta")
    .mode("append")
    .save(quarantine_path)
)

print("❌ Datos escritos en capa Quarantine")
print(f"Ruta: {quarantine_path}")

[Stage 41:========================================>               (36 + 2) / 50]

❌ Datos escritos en capa Quarantine
Ruta: /app/data/lakehouse/quarantine/secop_errors


In [8]:
spark.sql("""
DESCRIBE HISTORY delta.`/app/data/lakehouse/quarantine/secop_errors`
""").show(truncate=False)


+-------+---------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp            |userId|userName|operation|operationParameters                |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                   |userMetadata|engineInfo                         |
+-------+---------------------+------+--------+---------+-----------------------------------+----+--------+---------+-----------+--------------+-------------+-------------------------------------------------------------------+------------+-----------------------------------+
|0      |2026-02-01 21:31:33.7|NULL  |NULL    |WRITE    |{mode -> Append, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable  |true         |{numFiles -> 2

In [9]:
df_quarantine.groupBy("motivo_rechazo").count().orderBy("count", ascending=False).show(truncate=False)
# Los principales errores provienen de fechas nulas y valores monetarios inválidos

+---------------------------------------------------+-----+
|motivo_rechazo                                     |count|
+---------------------------------------------------+-----+
|Fecha inicio contrato nula                         |54730|
|Valor del contrato <= 0, Fecha inicio contrato nula|36463|
|Valor del contrato <= 0                            |865  |
+---------------------------------------------------+-----+



In [11]:
spark.stop()